# 🚀 IndoBERT Modeling & XAI Visualization

Notebook ini melatih model IndoBERT dan menyajikan visualisasi performa serta interpretasi model menggunakan XAI.

**Daftar Gambar untuk Laporan:**
1. `fig4_5_training_loss.png` - Kurva Loss Pelatihan.
2. `fig4_6_confusion_matrix.png` - Heatmap Confusion Matrix.
3. `fig4_7_lime_explanation.png` - Interpretasi LIME.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch pandas scikit-learn matplotlib seaborn lime shap

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix

# Config Path
BASE_PATH = "/content/drive/MyDrive/xai_lime_vs_shap"
TRAIN_DATA = os.path.join(BASE_PATH, "data/processed/train_balanced.csv")
TEST_DATA = os.path.join(BASE_PATH, "data/processed/test_original.csv")
FIG_DIR = os.path.join(BASE_PATH, "outputs/figures")
MODEL_SAVE_PATH = os.path.join(BASE_PATH, "outputs/indobert_model_final")

os.makedirs(FIG_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, reviews, targets, tokenizer, max_len):
        self.reviews = reviews
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.reviews)
    def __getitem__(self, item):
        encoding = self.tokenizer(str(self.reviews[item]), max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'targets': torch.tensor(self.targets[item], dtype=torch.long)}

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
df_train = pd.read_csv(TRAIN_DATA)
df_test = pd.read_csv(TEST_DATA)

# Mapping label numerik jika belum ada
LABEL_MAP = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
df_train['label'] = df_train['sentiment_label'].map(LABEL_MAP)
df_test['label'] = df_test['sentiment_label'].map(LABEL_MAP)

train_loader = DataLoader(ReviewDataset(df_train.review_text_clean.to_numpy(), df_train.label.to_numpy(), tokenizer, 128), batch_size=16, shuffle=True)
test_loader = DataLoader(ReviewDataset(df_test.review_text_clean.to_numpy(), df_test.label.to_numpy(), tokenizer, 128), batch_size=16, shuffle=False)

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
loss_fn = torch.nn.CrossEntropyLoss().to(device)

history = {'train_loss': []}
for epoch in range(epochs):
    model.train()
    epoch_losses = []
    for batch in train_loader:
        ids, mask, targets = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['targets'].to(device)
        outputs = model(ids, attention_mask=mask)
        loss = loss_fn(outputs.logits, targets)
        epoch_losses.append(loss.item())
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    history['train_loss'].append(np.mean(epoch_losses))
    print(f"Epoch {epoch+1} Loss: {history['train_loss'][-1]:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], marker='o', label='Training Loss')
plt.title("Training Loss Curve")
plt.savefig(os.path.join(FIG_DIR, "fig4_5_training_loss.png"))
plt.show()

In [ ]:
model.eval()
y_pred, y_true = [], []
with torch.no_grad():
    for batch in test_loader:
        ids, mask, targets = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['targets'].to(device)
        preds = torch.max(model(ids, attention_mask=mask).logits, dim=1)[1]
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(targets.cpu().numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(LABEL_MAP.keys()), yticklabels=list(LABEL_MAP.keys()))
plt.title("Confusion Matrix Final")
plt.savefig(os.path.join(FIG_DIR, "fig4_6_confusion_matrix.png"))
plt.show()